In [ ]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import * 

In [ ]:
spark.version

In [ ]:
df = spark.read.csv("hdfs://localhost:9000/data/employee_sales.csv", header=True, inferSchema=True)
df.show(5)

In [ ]:
df.describe().show()

In [ ]:
df.printSchema()

In [ ]:
df.select("name", "department", "sales_amount")\
    .filter(("department == 'Electronics'")  and ("region == 'Cairo'"))\
    .orderBy(col("sales_amount").desc())\
    .show(5)

In [ ]:
df2 = df.withColumn(
    "sales_with_tax",
    round(col("sales_amount") * 1.14, 2)
)
df2.select("name", "department", 'sales_amount', "sales_with_tax").show(5)

In [ ]:
summary = df.groupBy("department")\
    .agg(
        sum("sales_amount").alias("total_sales"),
        count("sales_amount").alias("sales_count"),
        floor(avg("sales_amount").alias("avgerage_count"), 2))\
    .orderBy(col("sales_count").desc())
summary.show()

In [ ]:
df.createOrReplaceTempView("customers")

In [ ]:
spark.sql("SELECT * FROM customers LIMIT 5").show()

In [ ]:
spark.sql("""
    SELECT 
        department,
        SUM(sales_amount) AS total_sales,
        COUNT(sales_amount) AS count_sales,
        AVG(sales_amount) AS average_sales
    FROM customers 
    GROUP By department
    ORDER BY total_sales DESC
    """)\
    .show()

In [ ]:
spark.sql("""
    SELECT
        *,
        SUM(sales_amount) OVER(PARTITION BY department ORDER BY region) AS department_total_sales
    FROM customers
""").show(5)

In [ ]:
spark.sql("""
    SELECT 
        *,
        RANK() OVER(ORDER BY sales_amount DESC) AS sales_rank
    FROM customers
""").show(5)